# AoC 2024 Day 16 — Reindeer Maze

**Python — Dijkstra over (tile, heading)**

Puzzle: <https://adventofcode.com/2024/day/16>

---

> **On puzzle text and inputs.** Advent of Code is Eric Wastl's work, and he asks that puzzle text and per-user inputs not be redistributed. So this notebook carries a summary in my own words plus the *published* example, and pulls the real input at runtime from a local cache that is gitignored. Read the puzzle at the link above.

> **Part 1 only.** Advent of Code reveals Part Two only after a correct Part One submission, and this day is unsolved — so part 2's text does not exist to work from yet. Submitting the answer below unlocks it.

---

## The puzzle

A maze of walls (`#`) and open tiles (`.`), with a start `S` and an end `E`. A reindeer begins on `S` **facing East**.

Two moves are available, and they cost different amounts:

- step forward one tile — **1 point**
- rotate 90° clockwise or counterclockwise in place — **1000 points**

- **Part 1** — find the lowest total score of any route from `S` to `E`.

The 1000:1 ratio is the whole puzzle. A route that is longer in tiles but straighter can easily beat a shorter, twistier one, so this is not a shortest-path problem on the grid — it is a shortest-path problem on `(tile, heading)`.

## The approach

This one is Python, and the reason is worth being precise about.

Dijkstra is **sequentially dependent by construction**: you cannot settle the next state until you know which unsettled state currently has the lowest cost. That is a global minimum over the frontier, recomputed after every single pop. A local binary heap does it in `O(log n)` per step with no coordination.

The Spark framing exists — it is iterative join-based relaxation. Keep a `dist(r, c, h)` frame, join it to a transitions frame, `groupBy` to take the min cost per state, and loop until nothing improves. Each iteration is a **shuffle plus an action to test the fixpoint**, so it is one Spark job per frontier hop. The real maze — 141×141, ~10000 open tiles, ~40000 settled states — runs hundreds of rounds before the costs stop changing, each paying scheduler latency and a full shuffle of a frame with a few thousand rows.

So the ledger is: hundreds of round-trips to a cluster, versus ~70 ms of heap operations on the driver, for the same number. There is no data-size argument on the other side — the whole state space is `open tiles × 4 headings`, low tens of thousands of rows. Spark would be paying distribution costs to solve a problem that never needed distributing.

## Setup

Connect to the cluster's Spark Connect endpoint and import the solution.

In [ ]:
import sys

sys.path.insert(0, '..')  # so `aoc_spark` resolves when running from notebooks/

from aoc_spark.session import get_spark
from aoc_spark.inputs import get_input
from aoc_spark.y2024 import day16

spark = get_spark('aoc-2024-day16')
print('Spark', spark.version)

## The published example

The same data the test suite asserts on.

In [ ]:
EXAMPLE = '###############\n#.......#....E#\n#.#.###.#.###.#\n#.....#.#...#.#\n#.###.#####.#.#\n#.#.#.......#.#\n#.#.#####.###.#\n#...........#.#\n###.#.#####.#.#\n#...#.....#.#.#\n#.#.#.###.#.#.#\n#.....#...#.#.#\n#.###.#.#.#.#.#\n#S..#.....#...#\n###############\n'

print('part 1:', day16.part1(spark, EXAMPLE), '(expected 7036)')

### The cost of distributing this

The cell below runs the same Dijkstra as `part1`, but counts the frontier hops. Each distinct cost level is a point where the algorithm needs a global minimum over the whole frontier — a coordination barrier a distributed version would pay a job for. The whole search finishes on the driver in milliseconds; the real maze takes ~70 ms.

In [ ]:
import heapq
import time

grid = EXAMPLE.strip('\n').splitlines()
start = next((r, c) for r, row in enumerate(grid) for c, ch in enumerate(row) if ch == 'S')
end = next((r, c) for r, row in enumerate(grid) for c, ch in enumerate(row) if ch == 'E')
open_tiles = sum(row.count('.') for row in grid) + 2  # S and E are open too
print(f'{len(grid)}x{len(grid[0])} maze, {open_tiles} open tiles -> {open_tiles * 4} (tile, heading) states')

# The same Dijkstra as part1, instrumented: every distinct cost pulled off
# the heap is one frontier hop, i.e. one Spark job in the iterative-join framing.
started = time.perf_counter()
queue = [(0, start[0], start[1], 0)]
best = {}
hops, last_cost, cost = 0, -1, None
while queue:
    cost, r, c, h = heapq.heappop(queue)
    if (r, c) == end:
        break
    if best.get((r, c, h), cost + 1) <= cost:
        continue
    if cost != last_cost:
        hops, last_cost = hops + 1, cost
    best[(r, c, h)] = cost
    dr, dc = day16.HEADINGS[h]
    if grid[r + dr][c + dc] != '#':
        heapq.heappush(queue, (cost + 1, r + dr, c + dc, h))
    for turn in (1, 3):
        heapq.heappush(queue, (cost + day16.TURN, r, c, (h + turn) % 4))
elapsed = (time.perf_counter() - started) * 1000

print(f'settled {len(best)} states over {hops} distinct cost levels')
print(f'answer {cost} in {elapsed:.1f} ms on the driver')
print(f'-> {hops} coordination points: each is a global minimum over the frontier,')
print('   which is one shuffle + one action if the frontier lives in a DataFrame')

## The real input

`get_input` is cache-first: local gitignored file → Postgres → adventofcode.com. In practice it hits the local file and never touches the network.

In [ ]:
import time

data = get_input(2024, 16)
print(f'input: {len(data):,} chars, {len(data.splitlines()):,} lines')

started = time.perf_counter()
answer = day16.part1(spark, data)
print(f'part 1: {answer}  ({(time.perf_counter() - started) * 1000:.0f} ms)')

## Cross-check

Days 6+ have no known-good answer, so correctness rests on an independently written plain-Python implementation agreeing with the Spark one. That is evidence, not proof — a shared misreading of the puzzle would survive both.

In [ ]:
from reference_python.y2024 import day16 as reference

cross = reference.part1(data)
print('reference:', cross)
print('agree:    ', cross == answer)

## Notes & gotchas

- The state is **`(row, col, heading)`, not `(row, col)`**. Arriving at a tile facing north is a different state from arriving facing east, because the 1000-point turn cost you owe next depends on it. Collapsing the heading out of the key gives wrong answers on mazes where the cheap approach faces the wrong way.
- `HEADINGS[0]` is East, matching the puzzle's "the Reindeer starts facing East". Reorder that list and the start orientation silently changes.
- Turns are pushed **unconditionally**, without checking whether the new heading faces a wall. That is safe — a turn into a wall just settles a state that can never step forward — and it keeps the inner loop branch-free. Only two turns are pushed (`+1` and `+3`); a 180° reversal is reachable as two 90° turns at the same cost, so enumerating it would be redundant.
- The `best.get(..., cost + 1) <= cost` guard is the lazy-deletion idiom: stale heap entries are skipped on pop rather than removed on push. Without it the loop still terminates but re-expands settled states.
- Returning on **pop** of the end tile, not on push, is what makes the first hit optimal. Checking at push time would return the first path found, not the cheapest.
- The walk indexes `grid[r + dr][c + dc]` with no bounds check. That is safe only because AoC mazes are fully wall-bordered — the reindeer can never step off the edge. An unbordered grid would raise `IndexError` or, worse, wrap on a negative index.